# DCT Laboratory — Volume II, Chapter 16
## Enterprise Applications and Integrated Transformation Case Studies
**Seed `26216`** · Companion to Chapter 16 and **AXIOM Lab 2.16** (module **AXIOM-16**) · Mirrored in `DCT_V2_Ch16_Lab.xlsx`

This laboratory reproduces **Worked Example 16.1, *Can Meridian Run Its Whole Program?***, with the book's
declarations, and works the five steps of AXIOM Lab 2.16: check jointly, order, price, keep a choice open, and certify.
The pieces are those Volume II certified one chapter at a time: the build's law and levers (Volume I's law on Chapter
9's 100,000 paths, drawn from Volume I, Chapter 8's stream with seed 8, and checked on the book's 100,000 fresh paths,
seed 26209), the sale with the engineers kept, Chapter 5's automation program, Chapter 11's family of laws (its extra
closures from seed 1108) and Volume I's feedback core. The new declarations put them on one account (Online Appendix
Remark 16.B.1). The book's full search over 64 schedules of rounds, payouts and packages takes minutes; this notebook
evaluates the printed programs exactly, each cheapest package a quantile of the paths' critical depths, and confirms
that they are best nearby. The lab's own seed, 26216, draws only Panel 6's 100,000 new paths. The cells take about a
minute on two cores. Every Meridian number below is the book's (Tables 16.1 and 16.3, Figures 16.1–16.3, Online
Appendix Example 16.B.2); the final cell checks them. Money is in \$ million.

## The declarations (Worked Example 16.1, Step 1; Table 16.1; Online Appendix Remark 16.B.1)

**The pieces, as their chapters declared them (Table 16.1).** *The build and its package* (Volume I, Worked Example
15.1; Chapters 1, 4, 9–11, 13). Volume I's plan of liquidity over the build's twelve quarters is the plan without the
draw, $B_k$, less a draw of depth $\delta$ index points with shape $S_k$, \$6.2 million a point at its deepest, two
years in. Execution deviations move it, $E' = 0.6E + w$ with spread \$50 million a quarter, and funding windows close
at 0.10 a year, each removing a gamma-distributed amount (mean \$120 million, standard deviation \$50 million) that is
never recovered. Two levers: a standby **line** $l$ that covers up to $l$ of each closure, for fees of $0.03\,l$ over
the program, and $\phi$ quarters of committed **delay**, $\delta = 28 - 4\phi$, at \$8 million a quarter. The board's
**appetite**: at most a 5 percent chance that liquidity falls below the \$350 million floor in some quarter of the
build. Volume I's package, a \$170 million line and 0.668 quarter, \$10.44 million, keeps it for the build alone.
*The sale* (Volume I, Worked Example 3.1; Chapters 6–7): \$112 million with the shared engineering team, \$92 million
if Meridian keeps it; Chapters 6–7 priced keeping the team at \$23.8 million against the buyer's \$20 million discount,
so the sale keeps the engineers. *The automation rounds and payouts* (Volume I, Worked Example 6.1; Chapters 2–5, 12):
a round costs \$120 million at the start of its year and halves the gap from efficiency 71 to the ceiling of 85, each
point saving \$10 million a year; cash stays at or above \$350 million at year ends, \$470 million before a round and
\$470 million at the end of year 6; the program's value is the payouts at the 10 percent hurdle rate plus the year-6
savings forever (Chapter 5). *The capability rule and twin* (Chapters 6–8, 14–15) keep capability at or above 55,
which the build and the rounds need; each round retrains staff and adds a point.

**The new declarations (Worked Example 16.1, Step 1).** *One account* (the treasury): liquidity is Volume I's plan for
the build plus the automation program's cash and the sale's proceeds, $L_k = B_k - \delta S_k + E_k - U_k(l) + G + A_k$
at the end of quarter $k$, where $U_k(l)$ is the closures the line leaves uncovered, $G$ the sale's proceeds and $A_k$
the automation program's cash so far. Volume I's Worked Example 6.1 had assumed the rest of Meridian's cash flows left
liquidity unchanged. *Timing* (the COO and the CFO): a round is paid at the start of its year, savings accrue evenly
over the year, and a payout is made at the year's end, after the quarter's reading, so it first counts the next
quarter. *The sale* (the board) closes at the start, with the engineers kept: $G = 92$. *The appetite* (the board)
still counts the build's twelve quarters; after the build (years 4–6) the automation program keeps Chapter 5's
commitments on the planned path, with Volume I's plan continued by its own shapes and the delayed spending caught up
(depth 28 from year 4). *The value* (the board): Chapter 5's, $\sum_{j=1}^{6}\beta^{j}p_j + \beta^{6}S_6/0.10$ with
$\beta = 1/1.1$ and $S_6$ the year-6 savings, less the levers' cost $0.03\,l + 8\phi$.

**How a program is checked (Remark 16.B.1).** For a line $l$ and a program's cash $G + A_k$, a path keeps the floor in
every quarter exactly when the draw is no deeper than the path's **critical depth**,
$\min_k\,(B_k + E_k - U_k(l) + G + A_k - 350)/S_k$. The chance of a breach is a count over the paths, and the least
delay that meets an appetite is a quantile of the critical depths, as in Chapter 9. The cheapest package tries lines
every \$20 million up to \$400 million, then every \$5 million within \$15 million of the best.

In [ ]:
import itertools
import math
import time
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
T_START = time.time()
SEED = 26216        # the lab's seed: it draws only Panel 6's 100,000 new paths
BOOK_SEED = 8       # Volume I, Chapter 8's stream: the book's 100,000 paths (Chapter 9)
FRESH_SEED = 26209  # the book's 100,000 fresh paths, on which every answer is checked
EXTRA_SEED = 1108   # Chapter 11's family on the book's paths: the extra closures and the thinning uniforms

N, Q = 100_000, 12                          # paths; quarters of the build
# Volume I's law (Chapter 8) in USD billion, exactly as its engine draws it; every result is reported in $ million
A_E, SD_W = 0.6, 0.05                       # execution deviations E' = 0.6 E + w, w with spread $50m a quarter
LAM, J_MEAN, J_SD = 0.10, 0.12, 0.05        # funding-window closures a year; gamma sizes, mean $120m and sd $50m
GAMMA = ((J_MEAN / J_SD) ** 2, J_SD ** 2 / J_MEAN)      # the sizes' shape and scale
LAM_MAX = 0.40                              # Chapter 11: the rate ceiling of the extra closures
# the build's levers and the board's appetite (Volume I, Worked Example 15.1; Chapter 9)
FLOOR, LIQ0 = 350.0, 620.0                  # the floor; liquidity today
ROOM = LIQ0 - FLOOR                         # the room above the floor
DEPTH0, DPQ = 28.0, 4.0                     # the draw's depth (index points); a quarter of delay makes it 4 points shallower
FEE, VQ, ALPHA = 0.03, 8.0, 0.05            # line fees over the program; a quarter of delay; the appetite
PKG_VOL1 = (170.0, 0.667576361426625)       # Volume I's package, as Chapter 9 designed it on the book's paths
# the sale (Volume I, Worked Example 3.1; Chapters 6-7) and Chapter 5's automation program
SALE = 112.0 - 20.0                         # the sale's proceeds with the engineers kept
ROUND, E0, CEIL, VAL = 120.0, 71.0, 85.0, 10.0   # a round's cost; efficiency today; its ceiling; savings a point a year
PRE, END = 470.0, 470.0                     # cash before a round; cash at the end of year 6
R = 0.10                                    # the board's hurdle rate
BY = 1.0 / (1.0 + R)                        # a year's discount
CAP_PRICE = 0.975 * 4.0                     # Chapter 7: a point of capability on the floor, in hiring
CLOSURE_P90 = 1000.0 * stats.gamma.ppf(0.9, GAMMA[0], scale=GAMMA[1])   # Volume I's closure buffer, the 90th percentile

def relax(x0, x_inf, k, t):
    return x_inf + (x0 - x_inf) * np.exp(-k * t)

def trough(depth, tau, t):
    return depth * (t / tau) * np.exp(1.0 - t / tau)

TQ = np.arange(1, Q + 1) / 4.0                                     # the twelve quarters, in years
BASE = (0.62 * relax(100.0, 112.0, 0.35, TQ) / 100.0) * 1000.0     # Volume I's plan without the draw (index 100 = $620m)
SHAPE = (0.62 * trough(1.0, 2.0, TQ) / 100.0) * 1000.0            # the draw per index point of depth
INV_SHAPE = 1.0 / SHAPE

def plan_level(t, depth=DEPTH0):
    # Volume I's planned liquidity t years from now, continued by its own shapes
    return 1000.0 * 0.62 * (relax(100.0, 112.0, 0.35, t) - trough(depth, 2.0, t)) / 100.0

B_YEARS = [plan_level(float(j)) for j in range(7)]       # year ends 0-6 on the planned path, depth 28 after the build

def draws(seed, n=N):
    # Volume I's stream in its order: each quarter the execution shocks, the closure counts, then the sizes path by path
    rng = np.random.default_rng(seed)
    W, events = np.empty((n, Q)), []
    for k in range(Q):
        W[:, k] = rng.normal(0.0, SD_W, n)
        count = rng.poisson(LAM / 4.0, n)
        events.append((np.repeat(np.arange(n), count), rng.gamma(*GAMMA, int(count.sum()))))
    E, e = np.zeros((n, Q)), np.zeros(n)
    for k in range(Q):
        e = A_E * e + W[:, k]
        E[:, k] = e
    return E, events

def cost(line, phi):
    return FEE * line + VQ * phi

class Paths:
    # The paths of one law, with the one-account check of Remark 16.B.1. For a line and a program's cash G + A_k a path
    # keeps the floor in every quarter exactly when the depth is at most its critical depth, so chances are exact counts
    # and the least delay meeting an appetite is a quantile of the critical depths.
    def __init__(self, E, events, seed):
        self.E, self.events, self.seed, self.n = E, events, seed, E.shape[0]
    @classmethod
    def draw(cls, seed, n=N):
        return cls(*draws(seed, n), seed)
    def lost(self, line):                   # cumulative closures the line does not cover (USD bn), quarter by quarter
        J, j = np.zeros((self.n, Q)), np.zeros(self.n)
        for k, (path, size) in enumerate(self.events):
            j = j + np.bincount(path, np.maximum(size - line / 1000.0, 0.0), self.n)
            J[:, k] = j
        return J
    def ratio(self, line):                  # (plan without the draw + deviations - uncovered closures - floor) / shape
        return (BASE[None, :] + 1000.0 * (self.E - self.lost(line)) - FLOOR) * INV_SHAPE[None, :]
    def crit(self, line, A, G):             # each path's critical depth with the program's cash G + A_k
        return np.min(self.ratio(line) + ((A + G) * INV_SHAPE)[None, :], axis=1)
    def depth_for(self, line, A, G, p=ALPHA):
        c = self.crit(line, A, G)
        j = int(np.floor(p * c.size + 1e-9))
        return min(DEPTH0, float(np.partition(c, j)[j]))
    def chance(self, line, phi, A, G):      # the chance of a breach in some quarter of the build
        return float(np.mean(self.crit(line, A, G) < DEPTH0 - DPQ * phi))
    def liquidity(self, line, phi, A, G):   # every path's liquidity at each quarter end ($m)
        return BASE[None, :] - (DEPTH0 - DPQ * phi) * SHAPE[None, :] + 1000.0 * (self.E - self.lost(line)) + G + A[None, :]
    def cheapest(self, A, G, p=ALPHA, coarse=20.0, fine=5.0):
        # the cheapest committed package keeping the appetite p: (cost, line, delay), or None
        best = None
        def try_line(line):
            nonlocal best
            d = self.depth_for(line, A, G, p)
            if d < 0:
                return
            phi = (DEPTH0 - d) / DPQ
            if best is None or cost(line, phi) < best[0] - 1e-12:
                best = (float(cost(line, phi)), float(line), float(phi))
        for line in np.arange(0.0, 400.01, coarse):
            try_line(line)
        if best is None:
            return None
        l0 = best[1]
        for line in np.arange(max(0.0, l0 - coarse + fine), min(400.0, l0 + coarse - fine) + 1e-9, fine):
            try_line(line)
        return best

# Chapter 5's automation program
def efficiency(z):
    e, out = E0, []
    for zk in z:
        if zk:
            e = e + 0.5 * (CEIL - e)
        out.append(e)
    return np.array(out)

def savings(z):
    return VAL * (efficiency(z) - E0)

def war_chest(z, S):
    # the least year-end cash from which the rest of the schedule keeps every commitment with no further payout
    T = len(z)
    W_ = np.empty(T + 1)
    W_[T] = max(END, FLOOR)
    for j in range(T, 0, -1):
        need = W_[j] + ROUND * z[j - 1] - S[j - 1]
        if z[j - 1]:
            need = max(need, PRE)
        if j - 1 >= 1:
            need = max(need, FLOOR)
        W_[j - 1] = need
    return W_

def ch5_plan(z, q0=LIQ0):
    # Chapter 5 on its own account: pay out everything above the war chest
    S = savings(z)
    W_ = war_chest(z, S)
    q, pays, qs, ok = q0, [], [], q0 >= W_[0] - 1e-9
    for j in range(1, len(z) + 1):
        if z[j - 1] and q < PRE - 1e-9:
            ok = False
        q = q - ROUND * z[j - 1] + S[j - 1]
        p = max(0.0, q - W_[j])
        q -= p
        pays.append(p); qs.append(q)
        ok = ok and q >= FLOOR - 1e-9
    ok = ok and q >= END - 1e-9
    val = sum(BY ** (k + 1) * pays[k] for k in range(len(z))) + BY ** len(z) * S[-1] / R
    return ok, np.array(pays), np.array(qs), W_, float(val)

# the program on one account
def automation_profile(z, p1, p2):
    # the automation program's cash at the end of quarters 1-12: rounds at year starts, savings evenly over the year,
    # payouts at year ends after the reading (they count from the next quarter)
    S = savings(z)
    A, cum, pays = np.zeros(Q), 0.0, (p1, p2, 0.0)
    for j in range(3):
        for mq in range(4):
            A[4 * j + mq] = cum - ROUND * z[j] + (mq + 1) * S[j] / 4.0
        cum = cum - ROUND * z[j] + S[j] - pays[j]
    return A

def post_window(z, p1, p2, G, G_late=0.0):
    # years 3-6 on the planned path: the war chest and the payouts p3..p6; None if a commitment fails
    S, B = savings(z), B_YEARS
    W_ = np.zeros(7)
    W_[6] = END
    for j in range(6, 3, -1):
        need = W_[j] + ROUND * z[j - 1] - S[j - 1] - (B[j] - B[j - 1]) - (G_late if j == 4 else 0.0)
        if z[j - 1]:
            need = max(need, PRE)
        W_[j - 1] = max(need, FLOOR)
    cash = B[3] + G + sum(-ROUND * z[i] + S[i] for i in range(3)) - p1 - p2
    pays, qs = [], []
    for j in range(3, 7):
        if j > 3:
            if z[j - 1] and cash < PRE - 1e-9:
                return None
            cash = cash - ROUND * z[j - 1] + S[j - 1] + (B[j] - B[j - 1]) + (G_late if j == 4 else 0.0)
        pj = max(0.0, cash - W_[j])
        cash -= pj
        pays.append(pj); qs.append(cash)
        if cash < FLOOR - 1e-9:
            return None
    if cash < END - 1e-9:
        return None
    return pays, qs, W_

def value(z, p1, p2, levers, G, G_late=0.0):
    # the program's value: payouts at 10 percent plus the year-6 savings forever, less the levers; (value, payouts)
    S = savings(z)
    pw = post_window(z, p1, p2, G, G_late)
    if pw is None:
        return None
    pays = [p1, p2] + pw[0]
    return float(sum(BY ** (j + 1) * pays[j] for j in range(6)) + BY ** 6 * S[-1] / R - levers), pays

def margins(z, p1, p2, G, phi):
    # planned margin above the floor by quarter: no deviations, no closures
    return BASE - (DEPTH0 - DPQ * phi) * SHAPE + G + automation_profile(z, p1, p2) - FLOOR

def claims(z, p1, p2, line, phi, G):
    # the deterministic joint check at the binding quarter: the claims on the room above the floor
    mg = margins(z, p1, p2, G, phi)
    k = int(np.argmin(mg))
    build = LIQ0 - (BASE[k] - (DEPTH0 - DPQ * phi) * SHAPE[k])
    paid = (p1 if k >= 4 else 0.0) + (p2 if k >= 8 else 0.0)
    rounds = -(automation_profile(z, p1, p2)[k] + paid)
    buffer = max(CLOSURE_P90 - line, 0.0)
    return dict(quarter=k + 1, build=build, rounds=rounds, payouts=paid, buffer=buffer, room=ROOM + G,
                left=ROOM + G - build - rounds - paid - buffer)

def rounds_of(z):
    return ", ".join(str(k + 1) for k in range(6) if z[k])

BOOK, FRESH = Paths.draw(BOOK_SEED), Paths.draw(FRESH_SEED)
NO_CASH = np.zeros(Q)
print(f"Volume I's plan without the draw: {BASE[0]:.1f} at quarter 1 rising to {BASE[-1]:.1f} at quarter 12, relaxing "
      f"toward {1000 * 0.62 * 112 / 100:.1f}; the draw {SHAPE.max():.2f} a point at its deepest (quarter {SHAPE.argmax() + 1})")
print(f"the book's paths (seed {BOOK_SEED}): {sum(len(p) for p, _ in BOOK.events):,} closures on 100,000 paths; the fresh "
      f"paths (seed {FRESH_SEED}): {sum(len(p) for p, _ in FRESH.events):,}")
print(f"Volume I's package for the build alone: line {PKG_VOL1[0]:.0f}, delay {PKG_VOL1[1]:.3f}, {cost(*PKG_VOL1):.2f}; "
      f"chance of a breach {100 * BOOK.chance(*PKG_VOL1, NO_CASH, 0.0):.2f}% (fresh {100 * FRESH.chance(*PKG_VOL1, NO_CASH, 0.0):.2f}%)")
print(f"Volume I's closure buffer, the 90th-percentile closure: {CLOSURE_P90:.1f}; the room above the floor: {ROOM:.0f}")

## Panel 1 — Declare one program, on one account (Worked Example 16.1, Step 1; Table 16.1; Online Appendix Remark 16.B.1 and Example 16.B.2: Chapter 5's plan alone runs rounds in years 1–3 with payouts 85, 0, 122.5, 122.5, 122.5, 2.5, worth \$1,021.93m; the board's price of \$5.645 for a dollar a year of savings; the draw \$6.2m a point at its deepest, the plan continued toward \$694m; the build alone draws \$121.6m at quarter 7 against a closure buffer of \$16.9m)

Worked Example 16.1, Step 1. Chapter 5's best plan, on its own account, pays out everything above its **war chest**,
the least year-end cash from which the rest of the schedule keeps every commitment with no further payout. On one
account the same plan joins Volume I's plan for the build and the sale's proceeds: quarter by quarter, the account
holds Volume I's planned liquidity, the automation program's cash so far and the \$92 million. After the build the plan
continues by its own shapes, toward \$694 million. The build alone, with Volume I's package, passes Volume I's joint
check at its deepest quarter: the draw and the closure buffer the line leaves uncovered fit the \$270 million of room.

In [ ]:
CH5 = (1, 1, 1, 0, 0, 0)                                  # Chapter 5's best plan: rounds in years 1-3
ok5, PAY5, CASH5, WAR5, V5 = ch5_plan(CH5)
S5 = savings(CH5)
PV5 = sum(BY ** (k + 1) * PAY5[k] for k in range(6))
SAV_PRICE = BY ** 6 / R                                   # the board's price of a dollar a year of savings (Chapter 12)
print("Chapter 5's plan alone, on an account where the rest of Meridian's cash flows leave liquidity unchanged:")
print("  year                " + "".join(f"{j:9d}" for j in range(1, 7)))
print("  round               " + "".join(f"{'yes' if z else '-':>9s}" for z in CH5))
print("  efficiency          " + "".join(f"{x:9.2f}" for x in efficiency(CH5)))
print("  savings             " + "".join(f"{x:9.1f}" for x in S5))
print("  war chest           " + "".join(f"{x:9.1f}" for x in WAR5[1:]))
print("  payout              " + "".join(f"{x:9.1f}" for x in PAY5))
print("  year-end cash       " + "".join(f"{x:9.1f}" for x in CASH5))
print(f"  value {V5:,.2f} = payouts {PV5:.2f} + the year-6 savings forever {BY ** 6 * S5[-1] / R:.2f} "
      f"(feasible: {ok5}); a dollar a year of savings is worth beta^6/0.10 = {SAV_PRICE:.3f}")

A5 = automation_profile(CH5, PAY5[0], PAY5[1])
D_PKG = DEPTH0 - DPQ * PKG_VOL1[1]
plan_pkg = BASE - D_PKG * SHAPE
print(f"\none account, quarter by quarter: Chapter 5's plan with the sale and Volume I's package (depth {D_PKG:.2f}):")
print("  quarter                    " + "".join(f"{k:7d}" for k in range(1, 13)))
print("  plan without the draw      " + "".join(f"{x:7.1f}" for x in BASE))
print("  draw per point of depth    " + "".join(f"{x:7.2f}" for x in SHAPE))
print("  Volume I's planned path    " + "".join(f"{x:7.1f}" for x in plan_pkg))
print("  automation cash so far     " + "".join(f"{x:7.1f}" for x in A5))
print("  one account, with the sale " + "".join(f"{x:7.1f}" for x in plan_pkg + A5 + SALE))
print(f"Volume I's plan at the year ends 0-6 (depth 28): " + ", ".join(f"{x:.1f}" for x in B_YEARS)
      + f"; it relaxes toward {1000 * 0.62 * 112 / 100:.1f}")
k7 = int(np.argmin(plan_pkg))
BUILD_ALONE = dict(quarter=k7 + 1, draw=LIQ0 - plan_pkg[k7], buffer=max(CLOSURE_P90 - PKG_VOL1[0], 0.0))
print(f"the build alone: its deepest draw {BUILD_ALONE['draw']:.1f} at quarter {BUILD_ALONE['quarter']}, plus the closure "
      f"buffer the line leaves uncovered {BUILD_ALONE['buffer']:.1f}, is {BUILD_ALONE['draw'] + BUILD_ALONE['buffer']:.1f} "
      f"of the {ROOM:.0f} of room (Volume I's joint check)")

## Panel 2 — Check jointly (Theorem 16.1(i); Worked Example 16.1, Step 2; Figure 16.1; Table 16.3; Online Appendix Example 16.B.2: signed together, Chapter 5's plan and Volume I's package breach the floor on 74.33% of paths with the sale (fresh 74.20%) and 99.58% without it (fresh 99.57%), 87.29% and 99.90% without levers; the planned margin at quarter 9 is \$7.3m and $-\$84.7$m; the claims at quarter 9, 115.4, 154.4, 85.0 and 16.9, exceed the room of 362 by 9.6, or of 270 by 101.6; keeping the pieces needs a \$165m line and 5.634 quarters, \$50.02m, and nothing keeps them without the sale; the program's claims 80.8, 154.4, 0 and 26.9 leave 100.0)

AXIOM Lab 2.16, step 1. Sign Chapter 5's plan and Volume I's package together, with the sale, on one account, and read
the chance of a breach on the book's paths and the fresh ones. Each piece passed its own check on the whole room: the
build keeps 5 percent alone, and Chapter 5's plan keeps its floor on an account where nothing else moves. Together they
claim the same room (Theorem 16.1(i)). The joint check at the binding quarter adds the claims on the room above the
floor: the build's draw, the rounds less their savings so far, the payouts already made, and the closure buffer the line
leaves uncovered. For comparison, the program the Worked Example signs (rounds in years 1–3, payouts held; Panels 3–4
find it best) is checked the same way.

In [ ]:
SEP = {}
for G in (0.0, SALE):
    SEP[G] = dict(chance=BOOK.chance(*PKG_VOL1, A5, G), fresh=FRESH.chance(*PKG_VOL1, A5, G),
                  no_levers=BOOK.chance(0.0, 0.0, A5, G), margins=margins(CH5, PAY5[0], PAY5[1], G, PKG_VOL1[1]),
                  cheapest=BOOK.cheapest(A5, G), claims=claims(CH5, PAY5[0], PAY5[1], *PKG_VOL1, G))
for G, lab in ((SALE, "with the sale"), (0.0, "without the sale")):
    s = SEP[G]
    print(f"pieces as certified, {lab:16s}: breach {100 * s['chance']:.2f}% (fresh {100 * s['fresh']:.2f}%), without "
          f"levers {100 * s['no_levers']:.2f}%; least planned margin {s['margins'].min():6.1f} at quarter "
          f"{int(np.argmin(s['margins'])) + 1}")
print("planned margin above the floor with the sale, by quarter: " + ", ".join(f"{x:.1f}" for x in SEP[SALE]["margins"]))
KEEP = SEP[SALE]["cheapest"]
print(f"keeping the pieces as certified, with the sale: line {KEEP[1]:.0f}, delay {KEEP[2]:.3f}, levers {KEEP[0]:.2f}, "
      f"breach {100 * BOOK.chance(KEEP[1], KEEP[2], A5, SALE):.2f}%")
best_depth = max(BOOK.depth_for(l, A5, 0.0) for l in np.arange(0.0, 400.01, 5.0))
print(f"without the sale no package keeps them (cheapest package: {SEP[0.0]['cheapest']}): even with a $400m line the "
      f"draw would have to be {best_depth:.2f} points deep, more than seven quarters of delay can do")

BEST_Z, BEST_P = (1, 1, 1, 0, 0, 0), (0.0, 0.0)          # the program the Worked Example signs (Step 4)
A_BEST = automation_profile(BEST_Z, *BEST_P)
PKG_BEST = BOOK.cheapest(A_BEST, SALE)
CLAIMS = dict(pieces_no_sale=SEP[0.0]["claims"], pieces=SEP[SALE]["claims"],
              program=claims(BEST_Z, *BEST_P, PKG_BEST[1], PKG_BEST[2], SALE))
print(f"\nthe joint check at the binding quarter (claims on the room above the floor):")
print(f"  {'case':22s} {'quarter':>7s} {'build draw':>10s} {'rounds net':>10s} {'payouts':>8s} {'buffer':>7s} "
      f"{'room':>6s} {'left':>7s}")
for key, lab in (("pieces_no_sale", "pieces, no sale"), ("pieces", "pieces, with sale"), ("program", "program")):
    c = CLAIMS[key]
    print(f"  {lab:22s} {c['quarter']:7d} {c['build']:10.1f} {c['rounds']:10.1f} {c['payouts']:8.1f} {c['buffer']:7.1f} "
          f"{c['room']:6.0f} {c['left']:7.1f}")
print("  with no deviations and no closures the plan itself keeps "
      f"{SEP[SALE]['margins'][8]:.1f} above the floor at quarter 9; the closure buffer of {CLAIMS['pieces']['buffer']:.1f} "
      f"is what takes the claims {-CLAIMS['pieces']['left']:.1f} past the room")

# Figure 16.1: liquidity on one account (plan and the level 5 percent of paths fall below), and the claims at quarter 9
def liquidity_band(z, p, line, phi, G):
    A = automation_profile(z, *p)
    L = BOOK.liquidity(line, phi, A, G)
    return BASE - (DEPTH0 - DPQ * phi) * SHAPE + G + A, np.quantile(L, 0.05, axis=0)
BANDS = dict(pieces=liquidity_band(CH5, (PAY5[0], PAY5[1]), *PKG_VOL1, SALE),
             program=liquidity_band(BEST_Z, BEST_P, PKG_BEST[1], PKG_BEST[2], SALE))
fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.6, 3.6), gridspec_kw={"width_ratios": [1, 1.05]})
q_ = np.arange(1, 13)
for key, col, lab in (("pieces", "#E65100", "pieces as certified"), ("program", "#1F4E79", "program")):
    ax.plot(q_, BANDS[key][0], color=col, lw=1.6, label=f"{lab}: plan")
    ax.plot(q_, BANDS[key][1], color=col, lw=1.3, ls="--", label=f"{lab}: 5% of paths below")
ax.axhline(FLOOR, color="#999999", lw=0.8, ls=":")
ax.text(12.3, 352, "floor", fontsize=8, color="#616161", ha="right", va="bottom")
ax.set(xlim=(1, 12.4), ylim=(150, 760), xticks=[1, 3, 5, 7, 9, 11], xlabel="quarter of the build",
       ylabel="liquidity (\\$ million)")
ax.legend(frameon=False, fontsize=7, loc="upper right", ncol=1)
ax.set_title("(a) one account, with the sale", fontsize=9)
cases = (("pieces_no_sale", "pieces,\nno sale"), ("pieces", "pieces,\nwith sale"), ("program", "program"))
parts = (("build", "#1F4E79", "build's draw"), ("rounds", "#2E7D32", "rounds, net"), ("payouts", "#E65100", "payouts made"),
         ("buffer", "#9E9E9E", "closure buffer"))
x_, bottom = np.arange(3), np.zeros(3)
for key, col, lab in parts:
    v = np.array([CLAIMS[c][key] for c, _ in cases])
    bx.bar(x_, v, 0.55, bottom=bottom, color=col, alpha=0.85, label=lab, lw=0)
    bottom += v
for room, lab, ls in ((ROOM, "room", ":"), (ROOM + SALE, "room with the sale", "--")):
    bx.axhline(room, color="#333333", lw=0.9, ls=ls)
    bx.text(2.75, room + 6, f"{lab}: {room:.0f}", fontsize=7.5, color="#333333", ha="right", va="bottom")
bx.set(xticks=x_, xlim=(-0.45, 2.8), ylim=(0, 560), ylabel="claims at quarter 9 (\\$ million)")
bx.set_xticklabels([lab for _, lab in cases], fontsize=8)
bx.legend(frameon=False, fontsize=7, loc="upper left", ncol=2, columnspacing=0.8, handlelength=1.0)
bx.set_title("(b) the joint check at quarter 9", fontsize=9)
fig.suptitle("The pieces fail together; the program fits (seed 26216)", fontsize=10.5)
plt.tight_layout(); plt.show()

## Panel 3 — Order the moves (Theorem 16.2(i)–(ii); Worked Example 16.1, Step 3; Figure 16.2(a); Table 16.3; Online Appendix Example 16.B.2: rounds in years 1–3 \$1,039.41m, years 1, 2, 4 \$1,036.80m, 1, 3, 4 \$1,022.57m, 1, 4, 5 \$1,001.59m, 2–4 \$953.58m, 2, 4, 5 \$937.60m, 3–5 \$883.16m, 4–6 \$821.11m; no sale \$938.95m with \$52.71m of levers, the sale after the build \$1,001.79m; the sale first is worth \$100.5m, \$69.1m paid out at the end of year 3 and \$31.3m of levers, \$37.6m more than after the build)

AXIOM Lab 2.16, step 2. **The sale.** Kept with its engineers, the sale only adds cash: capability stays at 58, above
the 55 the build and the rounds need, and \$92 million commutes with every other move's cash, so by Theorem 16.2(ii)
it goes first. Sold with them it would take capability to 53 and its order would matter (Volume I, Worked Example 3.1).
The three orders are priced on the book's paths: no sale, the sale after the build (its proceeds arrive in year 4,
after the appetite's window), and the sale first. **The rounds.** For each timing of the three rounds the book's
search re-optimizes the payouts of years 1–2 and the package; here each printed program is evaluated exactly at its
printed payouts. After the build the payouts follow the war chest on the planned path, and each timing of the first
three years is checked against all eight of its continuations in years 4–6: 64 schedules in all.

In [ ]:
DOMAIN = 55.0
print(f"capability after the sale: engineers kept 58 >= {DOMAIN:.0f}: {58 >= DOMAIN}; sold with them 53 >= {DOMAIN:.0f}: "
      f"{53 >= DOMAIN} (Volume I, Worked Example 3.1)")
PRINTED = [((1, 1, 1, 0, 0, 0), 0.0, 0.0), ((1, 1, 0, 1, 0, 0), 0.0, 77.0), ((1, 0, 1, 1, 0, 0), 35.0, 0.0),
           ((1, 0, 0, 1, 1, 0), 61.0, 65.0), ((0, 1, 1, 1, 0, 0), 0.0, 0.0), ((0, 1, 0, 1, 1, 0), 0.0, 48.0),
           ((0, 0, 1, 1, 1, 0), 8.0, 0.0), ((0, 0, 0, 1, 1, 1), 64.0, 20.0)]   # the printed programs: rounds, payouts y1-2
TAILS = list(itertools.product([0, 1], repeat=3))
PROGRAMS, GRID64 = [], []
for z, p1, p2 in PRINTED:
    pk = BOOK.cheapest(automation_profile(z, p1, p2), SALE)
    row = []
    for zt in TAILS:                                       # every continuation after the build, same package
        v = value(tuple(z[:3]) + zt, p1, p2, pk[0], SALE)
        row.append(None if v is None else v[0])
    GRID64.append(row)
    v, pays = value(z, p1, p2, pk[0], SALE)
    PROGRAMS.append(dict(z=z, rounds=rounds_of(z), p1=p1, p2=p2, cost=pk[0], line=pk[1], phi=pk[2], value=v, payouts=pays,
                         best_tail=bool(max(x for x in row if x is not None) <= v + 1e-9)))
BEST = PROGRAMS[0]
print(f"\n{'rounds':10s} {'payouts y1, y2':>15s} {'line':>6s} {'delay':>7s} {'levers':>7s} {'value':>10s} {'below best':>11s}"
      f"  payouts in years 3-6")
for p in PROGRAMS:
    print(f"{p['rounds']:10s} {p['p1']:7.0f} {p['p2']:7.0f} {p['line']:6.0f} {p['phi']:7.3f} {p['cost']:7.2f} "
          f"{p['value']:10.2f} {BEST['value'] - p['value']:11.2f}  " + ", ".join(f"{x:.1f}" for x in p["payouts"][2:]))
print("\nall 64 schedules at the printed payouts (value; '-' breaks a commitment after the build); rounds after the build:")
print(f"{'years 1-3':10s}" + "".join(f"{('none' if not any(t) else ', '.join(str(4 + i) for i in range(3) if t[i])):>10s}"
                                    for t in TAILS))
for p, row in zip(PROGRAMS, GRID64):
    print(f"{rounds_of(tuple(p['z'][:3]) + (0, 0, 0)) or 'none':10s}" + "".join(f"{x:10.2f}" if x is not None else f"{'-':>10s}"
                                                                         for x in row))
print("each printed timing is the best continuation of its first three years:", all(p["best_tail"] for p in PROGRAMS))

PKG_NOSALE = BOOK.cheapest(A_BEST, 0.0)
NOSALE = value(BEST_Z, *BEST_P, PKG_NOSALE[0], 0.0)
LATE = value(BEST_Z, *BEST_P, PKG_NOSALE[0], 0.0, G_late=SALE)
print(f"\norder of the sale (rounds in years 1-3, payouts held): sale first {BEST['value']:,.2f}; sale after the build "
      f"{LATE[0]:,.2f}; no sale {NOSALE[0]:,.2f} (line {PKG_NOSALE[1]:.0f}, delay {PKG_NOSALE[2]:.2f}, levers {PKG_NOSALE[0]:.2f})")
SALE_WORTH = BEST["value"] - NOSALE[0]
print(f"the sale first is worth {SALE_WORTH:.2f}: {SALE:.0f} beta^3 = {SALE * BY ** 3:.2f} paid out at the end of year 3, "
      f"plus {PKG_NOSALE[0] - BEST['cost']:.2f} of levers it makes unnecessary; after the build it is worth "
      f"{BEST['value'] - LATE[0]:.2f} less: paid out a year later ({SALE * (BY ** 3 - BY ** 4):.2f}) and no levers saved")
print(f"moving round 3 to year 4 costs {BEST['value'] - PROGRAMS[1]['value']:.2f}; starting the rounds in year 2 costs "
      f"{BEST['value'] - PROGRAMS[4]['value']:.2f}")

## Panel 4 — Price and allocate (Theorem 16.1(iii); Proposition 16.3; Worked Example 16.1, Step 4; Figure 16.2; Table 16.3; Online Appendix Example 16.B.2: the best program buys a \$160m line and 2.071 quarters, \$21.37m, worth \$1,039.41m at a breach chance of 5.00% (fresh 4.99%), payouts 0, 0, 189.9, 160.4, 158.8, 32.7; the appetite is worth \$1.83m a point, 1,038.43 at 4.5% and 1,040.25 at 5.5%, against Volume I's \$1.81m for the build alone; the line buys a point of chance for \$1.76m and the delay for \$2.29m, paying in year 1 earns \$0.77m and in year 2 \$0.67m a point; a dollar of room at the start is worth \$1.088; the rounds' points of capability, at \$3.90, are worth \$10.67m for years 1–3 and \$10.38m for years 1, 2, 4)

AXIOM Lab 2.16, step 3. At the best program, read the prices (Theorem 16.1(iii)). **The appetite's price**: re-optimize
the program at 4.5 and 5.5 percent and divide the change in value by the point between them. **Each lever's cost per
point of chance**: \$5 million more line, or 0.05 quarter more delay, and the chance it buys. **Holding the payouts**:
paying \$5 million at the end of year 1 or 2 instead of year 3 earns $\beta - \beta^3$ or $\beta^2 - \beta^3$ a dollar
and spends chance; a payout is held while that earns less than the appetite's price (Proposition 16.3(iii)). **A dollar
of room**: one more \$1 million of sale proceeds, the program re-optimized (Proposition 16.3(i)). Before reading the
prices, the cell confirms that the program is best nearby: payouts in years 1–2 within a few million, and lines \$10
million either side of its own. Last, the lab's question: release Chapter 5's year-1 payout of \$85 million instead
of holding it.

In [ ]:
CH_BEST, CH_BEST_F = BOOK.chance(BEST["line"], BEST["phi"], A_BEST, SALE), FRESH.chance(BEST["line"], BEST["phi"], A_BEST, SALE)
POST = post_window(BEST_Z, *BEST_P, SALE)
PV_PAY = sum(BY ** (j + 1) * BEST["payouts"][j] for j in range(6))
PV_SAV = BY ** 6 * savings(BEST_Z)[-1] / R
print(f"best program: rounds in years {BEST['rounds']}, payouts held in years 1-2; line {BEST['line']:.0f}, delay "
      f"{BEST['phi']:.3f}, levers {BEST['cost']:.2f}; breach {100 * CH_BEST:.2f}% (fresh {100 * CH_BEST_F:.2f}%)")
print(f"  payouts " + ", ".join(f"{x:.1f}" for x in BEST["payouts"]) + "; planned year-end cash in years 3-6 "
      + ", ".join(f"{x:.0f}" for x in POST[1]) + "; war chest " + ", ".join(f"{x:.0f}" for x in POST[2][3:]))
print(f"  value {BEST['value']:,.2f} = payouts {PV_PAY:.2f} + the year-6 savings forever {PV_SAV:.2f} - levers {BEST['cost']:.2f}")

NEAR = {}
for p1 in (0.0, 1.0, 2.0, 5.0):
    for p2 in (0.0, 1.0, 2.0, 5.0):
        pk = BOOK.cheapest(automation_profile(BEST_Z, p1, p2), SALE)
        NEAR[(p1, p2)] = value(BEST_Z, p1, p2, pk[0], SALE)[0]
print("\npayouts in years 1-2 near the best (value less the best's):")
print("  year 2 ->      " + "".join(f"{p2:8.0f}" for p2 in (0.0, 1.0, 2.0, 5.0)))
for p1 in (0.0, 1.0, 2.0, 5.0):
    print(f"  year 1 = {p1:3.0f}   " + "".join(f"{NEAR[(p1, p2)] - BEST['value']:8.3f}" for p2 in (0.0, 1.0, 2.0, 5.0)))
LINE_NEAR = []
for line in np.arange(BEST["line"] - 10.0, BEST["line"] + 10.01, 5.0):
    phi = (DEPTH0 - BOOK.depth_for(line, A_BEST, SALE)) / DPQ
    LINE_NEAR.append((line, phi, cost(line, phi)))
print("lines within $10m: " + "; ".join(f"{l:.0f}: delay {ph:.3f}, {c:.2f}" for l, ph, c in LINE_NEAR))
LOCAL_BEST = all(v <= BEST["value"] + 1e-9 for v in NEAR.values()) and min(LINE_NEAR, key=lambda r: r[2])[0] == BEST["line"]
print("the best program is best nearby:", LOCAL_BEST)

# the appetite's price: the program re-optimized at 4.5 and 5.5 percent (its schedule held; payouts checked nearby)
def program_at(p=ALPHA, G=SALE):
    out = {}
    for pp in ((0.0, 0.0), (1.0, 0.0), (0.0, 1.0)):
        pk = BOOK.cheapest(automation_profile(BEST_Z, *pp), G, p=p)
        out[pp] = (value(BEST_Z, *pp, pk[0], G)[0], pk)
    best_pp = max(out, key=lambda k: out[k][0])
    return dict(value=out[best_pp][0], payouts=best_pp, cost=out[best_pp][1][0], line=out[best_pp][1][1],
                phi=out[best_pp][1][2], held=best_pp == (0.0, 0.0))
APP_LO, APP_HI = program_at(0.045), program_at(0.055)
MU = (APP_HI["value"] - APP_LO["value"]) / 1.0
print(f"\nthe appetite: at 4.5% line {APP_LO['line']:.0f}, delay {APP_LO['phi']:.3f}, levers {APP_LO['cost']:.2f}, value "
      f"{APP_LO['value']:,.2f}; at 5.5% line {APP_HI['line']:.0f}, delay {APP_HI['phi']:.3f}, levers {APP_HI['cost']:.2f}, "
      f"value {APP_HI['value']:,.2f}; payouts held at both: {APP_LO['held'] and APP_HI['held']}")
print(f"  the appetite's price: {MU:.3f} a percentage point")
def vol1_design(p, P=BOOK):
    # Volume I's design for the build alone (Volume I, Worked Example 15.1): every line on the $5m grid
    best = None
    for line in LINES_ALL:
        c = np.sort(P.crit(line, NO_CASH, 0.0))
        d = min(DEPTH0, c[int(np.floor(p * N))])
        if d >= 0 and (best is None or cost(line, (DEPTH0 - d) / DPQ) < best[2]):
            best = (float(line), (DEPTH0 - d) / DPQ, cost(line, (DEPTH0 - d) / DPQ))
    return best
LINES_ALL = np.arange(0.0, 400.01, 5.0)
V1_LO, V1_HI = vol1_design(0.045), vol1_design(0.055)
MU_VOL1 = V1_LO[2] - V1_HI[2]
print(f"  Volume I's price for the build alone: {V1_LO[2]:.2f} at 4.5% less {V1_HI[2]:.2f} at 5.5% = {MU_VOL1:.3f} a point")

# each lever's cost per point of chance, and what paying early earns per point it spends
c0 = CH_BEST
dl, dphi, dp = 5.0, 0.05, 5.0
CHANCES = dict(c0=c0, line=BOOK.chance(BEST["line"] + dl, BEST["phi"], A_BEST, SALE),
               delay=BOOK.chance(BEST["line"], BEST["phi"] + dphi, A_BEST, SALE),
               pay1=BOOK.chance(BEST["line"], BEST["phi"], automation_profile(BEST_Z, dp, 0.0), SALE),
               pay2=BOOK.chance(BEST["line"], BEST["phi"], automation_profile(BEST_Z, 0.0, dp), SALE))
LEVER = dict(line=FEE * dl / (100 * (c0 - CHANCES["line"])), delay=VQ * dphi / (100 * (c0 - CHANCES["delay"])),
             pay1=dp * (BY - BY ** 3) / (100 * (CHANCES["pay1"] - c0)), pay2=dp * (BY ** 2 - BY ** 3) / (100 * (CHANCES["pay2"] - c0)))
print(f"\nlevers at the best program: $5m more line buys {100 * (c0 - CHANCES['line']):.3f} points of chance, "
      f"{LEVER['line']:.3f} a point; 0.05 quarter more delay buys {100 * (c0 - CHANCES['delay']):.4f} points, "
      f"{LEVER['delay']:.3f} a point")
print(f"paying early: a dollar in year 1 instead of year 3 earns beta - beta^3 = {BY - BY ** 3:.3f} and spends "
      f"{100 * (CHANCES['pay1'] - c0) / dp:.3f} points of chance: {LEVER['pay1']:.3f} a point; in year 2, "
      f"beta^2 - beta^3 = {BY ** 2 - BY ** 3:.3f} for {100 * (CHANCES['pay2'] - c0) / dp:.3f} points: {LEVER['pay2']:.3f} a point")
print(f"  levers in use trade near the appetite's price ({LEVER['line']:.2f} and {LEVER['delay']:.2f} around {MU:.2f}: the "
      f"line moves in $5m steps); early payouts earn less than it, so they wait")

# a dollar of room at the start: one more $1m of proceeds, re-optimized
ROOM1 = program_at(ALPHA, SALE + 1.0)
ROOM_DOLLAR = ROOM1["value"] - BEST["value"]
print(f"\na dollar of room at the start: {ROOM_DOLLAR:.3f} = {BY ** 3:.3f} paid out at the end of year 3 + "
      f"{BEST['cost'] - ROOM1['cost']:.3f} of levers it makes unnecessary (line {ROOM1['line']:.0f}, delay {ROOM1['phi']:.3f})")
CAP_CREDIT = {p["rounds"]: sum(CAP_PRICE * BY ** k for k in range(6) if p["z"][k]) for p in PROGRAMS}
print(f"capability: each round adds a point, worth {CAP_PRICE:.2f} of hiring at Chapter 7's price on the floor; in present "
      f"value {CAP_CREDIT['1, 2, 3']:.2f} for rounds in years 1-3 and {CAP_CREDIT['1, 2, 4']:.2f} for years 1, 2, 4")

# hold or release: Chapter 5 paid $85m at the end of year 1
A_REL = automation_profile(BEST_Z, PAY5[0], 0.0)
PKG_REL = BOOK.cheapest(A_REL, SALE)
REL = value(BEST_Z, PAY5[0], 0.0, PKG_REL[0], SALE)
print(f"\nrelease Chapter 5's year-1 payout of {PAY5[0]:.0f}: the appetite then needs line {PKG_REL[1]:.0f} and delay "
      f"{PKG_REL[2]:.3f}, levers {PKG_REL[0]:.2f}; the program is worth {REL[0]:,.2f}, {BEST['value'] - REL[0]:.2f} less "
      f"than holding it")

fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.6, 3.6))
short = [BEST["value"] - p["value"] for p in PROGRAMS]
cols = ["#1F4E79" if i == 0 else ("#2E7D32" if i == 1 else "#BDBDBD") for i in range(len(PROGRAMS))]
ax.bar(np.arange(len(PROGRAMS)), short, 0.6, color=cols, edgecolor="#616161", lw=0.4)
for i in (0, 1):
    ax.text(i, short[i] + 5, f"{short[i]:.1f}", fontsize=8, ha="center", va="bottom")
ax.set_xticks(np.arange(len(PROGRAMS)), [p["rounds"] for p in PROGRAMS], fontsize=7.5, rotation=40, ha="right",
              rotation_mode="anchor")
ax.set(ylim=(0, 240), xlabel="years of the three rounds", ylabel="value below the best (\\$ million)")
ax.set_title("(a) each timing of the rounds, re-optimized", fontsize=9)
names = ["line", "delay", "pay in\nyear 1", "pay in\nyear 2"]
vals = [LEVER["line"], LEVER["delay"], LEVER["pay1"], LEVER["pay2"]]
bx.bar(np.arange(4), vals, 0.6, color=["#1F4E79", "#1F4E79", "#E65100", "#E65100"], alpha=0.85, lw=0)
for i, v in enumerate(vals):
    bx.text(i, v + 0.06, f"{v:.2f}", fontsize=8, ha="center", va="bottom")
bx.axhline(MU, color="#333333", lw=1.0, ls="--")
bx.text(3.45, MU + 0.07, f"appetite {MU:.2f}", fontsize=8, ha="right", va="bottom")
bx.set_xticks(np.arange(4), names, fontsize=8)
bx.set(xlim=(-0.5, 3.5), ylim=(0, 3.0), ylabel="\\$ million per point of chance")
bx.set_title("(b) buy a point (blue), spend a point (orange)", fontsize=9)
fig.suptitle("Rounds in years 1–3; levers at the appetite's price (seed 26216)", fontsize=10.5)
plt.tight_layout(); plt.show()

## Panel 5 — Keep a choice open (Theorem 16.2(iii); Proposition 16.2; Worked Example 16.1, Step 5; Figure 16.3(b); Table 16.3; Online Appendix Example 16.B.2: round 3 in year 3 only if cash after the year-2 payout is at least \$490m, on 83.0% of paths (fresh 83.0%), else in year 4, with a \$155m line and 1.642 quarters, \$17.79m: worth \$1,040.76m, \$1.35m more than committing it to year 3, at a breach chance of 5.00% (fresh 5.00%); Chapter 7's generic project is worth 5.00 started now and 22.73 if it waits)

AXIOM Lab 2.16, step 4. Round 3 is irreversible, and its best timing depends on how the build has gone. Put it on a
trigger: at the end of year 2, after the year-2 payout (none at the best program), run round 3 in year 3 if cash is at
least the trigger, and otherwise in year 4. On each path the reading is the cash at the end of quarter 8, so a path runs
round 3 exactly when the draw is shallow enough; for each line on the \$5 million grid from \$100 million to \$260
million, the least delay that keeps the appetite under the rule is found by bisection on the depth, and the rule's value
is the share of paths that run round 3 in year 3 times that program's value, plus the rest times the year-4 program's,
less the levers. Every trigger from \$380 million to \$700 million by \$10 million is priced; only the paths whose
critical depth under either timing is below 28 can breach, so the count runs over those (the result is the same as over
all paths). Theorem 16.2(iii): the rule is worth at least the best committed timing, and more exactly when the best
timing depends on the reading. Proposition 16.2's waiting formula, on Chapter 7's generic project, closes the panel.

In [ ]:
t0 = time.time()
Z_RUN, Z_WAIT = (1, 1, 1, 0, 0, 0), (1, 1, 0, 1, 0, 0)
P1_RULE, P2_RULE = 0.0, 0.0                               # the best program's payouts in years 1-2
V_RUN = value(Z_RUN, P1_RULE, P2_RULE, 0.0, SALE)[0]      # gross values of the two timings
V_WAIT = value(Z_WAIT, P1_RULE, P2_RULE, 0.0, SALE)[0]
LINES3 = np.arange(100.0, 260.01, 5.0)

def rule_paths(P):
    # per line: the critical depths under both timings and the cash at the end of quarter 8 at depth 0 (paths that
    # can breach at a depth of at most 28 are kept for the count; the reading is kept for every path)
    A_run, A_wait = automation_profile(Z_RUN, P1_RULE, P2_RULE), automation_profile(Z_WAIT, P1_RULE, P2_RULE)
    out = []
    for line in LINES3:
        r = P.ratio(line)
        c_run = np.min(r + ((A_run + SALE) * INV_SHAPE)[None, :], axis=1)
        c_wait = np.min(r + ((A_wait + SALE) * INV_SHAPE)[None, :], axis=1)
        L8 = r[:, 7] * SHAPE[7] + FLOOR + SALE + A_run[7] - P2_RULE
        near = np.minimum(c_run, c_wait) < DEPTH0
        out.append((float(line), c_run[near], c_wait[near], L8[near], L8))
    return out

def rule_at(T, pre, n=N, p=ALPHA):
    # the best package for trigger T: round 3 in year 3 iff cash after the year-2 payout >= T, else in year 4
    best = None
    for line, cr, cw, l8, l8_all in pre:
        dT = (l8 - T) / SHAPE[7]                         # the path runs round 3 iff the depth is at most dT
        breach = lambda d: np.count_nonzero(np.where(d <= dT, cr < d, cw < d)) / n
        lo, hi = 0.0, DEPTH0
        if breach(lo) > p:
            continue
        if breach(hi) <= p:
            d = hi
        else:
            for _ in range(50):                          # bisection on the depth, as the engine does
                mid = 0.5 * (lo + hi)
                if breach(mid) <= p:
                    lo = mid
                else:
                    hi = mid
            d = lo
        share = float(np.mean(d <= (l8_all - T) / SHAPE[7]))
        phi = (DEPTH0 - d) / DPQ
        v = share * V_RUN + (1 - share) * V_WAIT - cost(line, phi)
        if best is None or v > best["value"]:
            best = dict(trigger=float(T), line=line, phi=phi, cost=cost(line, phi), chance=breach(d), share_run=share, value=v)
    return best

PRE3 = rule_paths(BOOK)
TRIGGERS = np.arange(380.0, 700.01, 10.0)
RULE = [rule_at(T, PRE3) for T in TRIGGERS]
RB = max(RULE, key=lambda r: r["value"])

def rule_check(P, rb):
    # the rule's chance of a breach and its share of paths running round 3 in year 3, on other paths
    A_run, A_wait = automation_profile(Z_RUN, P1_RULE, P2_RULE), automation_profile(Z_WAIT, P1_RULE, P2_RULE)
    d = DEPTH0 - DPQ * rb["phi"]
    L8 = P.ratio(rb["line"])[:, 7] * SHAPE[7] + FLOOR + SALE + A_run[7] - P2_RULE
    run = d <= (L8 - rb["trigger"]) / SHAPE[7]
    hit = np.where(run, P.crit(rb["line"], A_run, SALE) < d, P.crit(rb["line"], A_wait, SALE) < d)
    return float(hit.mean()), float(run.mean())
RB_FRESH = rule_check(FRESH, RB)
COMMIT3, COMMIT4 = PROGRAMS[0]["value"], PROGRAMS[1]["value"]
print(f"gross values: round 3 in year 3 {V_RUN:,.2f}, in year 4 {V_WAIT:,.2f} (payouts held in years 1-2); "
      f"{len(TRIGGERS)} triggers priced in {time.time() - t0:.1f} s")
print(f"{'trigger':>8s} {'line':>6s} {'delay':>7s} {'levers':>7s} {'year 3 on':>10s} {'value':>10s}")
for r in RULE:
    if r["trigger"] % 40 == 0 or 460 <= r["trigger"] <= 520:
        print(f"{r['trigger']:8.0f} {r['line']:6.0f} {r['phi']:7.3f} {r['cost']:7.2f} {100 * r['share_run']:9.1f}% "
              f"{r['value']:10.2f}" + ("   <- best" if r is RB else ""))
print(f"best trigger {RB['trigger']:.0f}: round 3 in year 3 on {100 * RB['share_run']:.1f}% of paths (fresh "
      f"{100 * RB_FRESH[1]:.1f}%), line {RB['line']:.0f}, delay {RB['phi']:.3f}, levers {RB['cost']:.2f}; breach "
      f"{100 * RB['chance']:.2f}% (fresh {100 * RB_FRESH[0]:.2f}%)")
print(f"value {RB['value']:,.2f}: {RB['value'] - COMMIT3:.2f} more than committing round 3 to year 3 ({COMMIT3:,.2f}) and "
      f"{RB['value'] - COMMIT4:.2f} more than committing it to year 4 ({COMMIT4:,.2f})")
print(f"Theorem 16.2(iii): the rule beats both committed timings ({RB['value'] > max(COMMIT3, COMMIT4)}), strictly, because "
      f"the best timing depends on the reading: year 3 on {100 * RB['share_run']:.0f}% of paths, year 4 on the rest")

# Proposition 16.2 on Chapter 7's generic project (Online Appendix 7.B): costs 100, worth 150 or 60 after a year
c_, V_, beta_, d_ = 100.0, np.array([150.0, 60.0]), 1 / 1.1, 0.0
N0 = V_.mean() - c_
N1 = beta_ * np.maximum(V_ - c_, 0).mean() - d_
loss_avoided, wait_cost = np.maximum(c_ - V_, 0).mean(), (1 - beta_) * np.maximum(V_ - c_, 0).mean() + d_
print(f"\nProposition 16.2: started now {N0:.2f}; started next year only if it pays {N1:.2f}; the expected loss waiting "
      f"avoids {loss_avoided:.2f} exceeds what waiting costs {wait_cost:.2f}, so it waits ({N1 > max(N0, 0.0)})")

## Panel 6 — Certify (Theorem 16.3; Proposition 16.1; Worked Example 16.1, Step 6; Figure 16.3(a); Table 16.3; Online Appendix Example 16.B.2: at 5% the program certifies a closure rate of 0.100 and a spread of \$50.0m, and breaches on 13.88% of paths at the records' corner; designed for 4.5%, 4, 3.5, 3, 2.5 and 2% it costs \$22.35m, 23.44, 24.65, 26.02, 27.50 and 29.30m, keeps 5% for closure rates to 0.192, 0.298 and 0.40 or more, and spreads to \$51.0m, 52.2, 53.5, 55.0, 56.5 and 58.4m; at the corner, rate 0.315 and spread \$61.6m, the program needs a \$200m line and 3.440 quarters, \$33.52m, worth \$1,027.26m; the core's radius 0.800 today, 1.025 with the new link, 0.841 with every link trimmed to 0.304, 0.609 if the trim comes first)

AXIOM Lab 2.16, step 5. **No slack** (Theorem 16.3(i)): the best program meets the appetite exactly at the declared
law, so a closure rate or a spread any higher breaks it. **A margin buys a region** (Theorem 16.3(ii)): re-design the
program for 4.5, 4, …, 2 percent, its schedule held and its payouts re-optimized, and find along Chapter 11's family
the largest closure rate (0.10 to 0.40, the family's simulation ceiling) and the largest spread (\$50 million to 1.5
times that) at which it keeps the board's 5 percent, each edge by bisection. Chapter 11's family: a higher rate adds the
closures of an extra stream thinned by uniforms (seed 1108), and a wider spread scales every deviation. **The records'
corner**: the treasury's and the engineering office's one-sided 95 percent bounds, a rate of 0.315 and a spread of
\$61.6 million; the program re-designed there, and the price of ambiguity. **Triggers** (Theorem 16.3(iii)): the
region's edges, the twin's monitor, and Volume I's feedback core, whose phases (Proposition 16.1) are checked with
Volume I's symmetric register. Last, **new draws from the lab's seed, 26216**: the signed program on 100,000 paths of
its own.

In [ ]:
t0 = time.time()
def extra_stream(P, seed=EXTRA_SEED):
    # Chapter 11 (Online Appendix Remark 11.B.1): each quarter, uniforms for the paths' own closures; then the extra
    # stream at rate 0.40 - 0.10 (counts, gamma sizes, uniforms)
    rng = np.random.default_rng(seed)
    own, more = [], []
    for k in range(Q):
        path, size = P.events[k]
        own.append(rng.random(len(path)))
        count = rng.poisson((LAM_MAX - LAM) / 4.0, P.n)
        jj = np.repeat(np.arange(P.n), count)
        more.append((jj, rng.gamma(*GAMMA, len(jj)), rng.random(len(jj))))
    return own, more
EXTRA = extra_stream(BOOK)

def law(P, lam=LAM, c=1.0):
    # the law with closures at lam a year and the spread c x $50m, on P's paths
    if abs(lam - LAM) < 1e-15:
        return Paths(P.E * c, P.events, P.seed)
    own, more = EXTRA
    events = []
    for k, (path, size) in enumerate(P.events):
        if lam < LAM:
            keep = own[k] < lam / LAM
            events.append((path[keep], size[keep]))
        else:
            jj, sz, u = more[k]
            add = u < (lam - LAM) / (LAM_MAX - LAM)
            events.append((np.concatenate([path, jj[add]]), np.concatenate([size, sz[add]])))
    return Paths(P.E * c, events, P.seed)

def boundary(f, lo, hi, p=ALPHA, iters=30):
    # the largest x in [lo, hi] with f(x) <= p, for f nondecreasing (bisection)
    if f(hi) <= p:
        return hi
    if f(lo) > p:
        return lo
    for _ in range(iters):
        mid = 0.5 * (lo + hi)
        if f(mid) <= p:
            lo = mid
        else:
            hi = mid
    return lo

LAM_U = stats.chi2.ppf(0.95, 6) / 40.0                    # Chapter 11: two closures in twenty years, one-sided 95%
SIG_U = 50.0 * math.sqrt(39 / stats.chi2.ppf(0.05, 39))  # forty quarters of deviations, 39 innovations
SIG_U2 = 50.0 * math.sqrt(79 / stats.chi2.ppf(0.05, 79))  # a record twice as long
CORNER_C = 61.6 / 50.0                                    # the records' corner as the book prints it

print("no slack at the declared law: the best program's chance "
      + ", ".join(f"{lab} {100 * law(BOOK, lam, c).chance(BEST['line'], BEST['phi'], A_BEST, SALE):.2f}%"
                  for lab, lam, c in (("as declared", LAM, 1.0), ("rate 0.105", 0.105, 1.0), ("spread 50.5", LAM, 1.01))))
MARGINS = []
for a in (0.05, 0.045, 0.04, 0.035, 0.03, 0.025, 0.02):
    d = dict(value=BEST["value"], cost=BEST["cost"], line=BEST["line"], phi=BEST["phi"], payouts=BEST_P) if a == ALPHA \
        else program_at(a)
    line, phi = d["line"], d["phi"]
    lam_bar = boundary(lambda x: law(BOOK, x, 1.0).chance(line, phi, A_BEST, SALE), LAM, LAM_MAX)
    c_bar = boundary(lambda x: law(BOOK, LAM, x).chance(line, phi, A_BEST, SALE), 1.0, 1.5)
    MARGINS.append(dict(appetite=a, value=d["value"], extra=BEST["value"] - d["value"], cost=d["cost"], line=line, phi=phi,
                        held=d["payouts"] == BEST_P, chance=BOOK.chance(line, phi, A_BEST, SALE),
                        fresh=FRESH.chance(line, phi, A_BEST, SALE), lam_bar=lam_bar, capped=lam_bar >= LAM_MAX - 1e-9,
                        sigma_bar=50.0 * c_bar, corner=law(BOOK, LAM_U, CORNER_C).chance(line, phi, A_BEST, SALE)))
print(f"\n{'design':>6s} {'line':>5s} {'delay':>6s} {'levers':>7s} {'value':>9s} {'margin cost':>11s} {'breach':>7s} "
      f"{'fresh':>6s} {'rate to':>8s} {'spread to':>9s} {'at the corner':>13s}")
for m in MARGINS:
    rate = ">= 0.40" if m["capped"] else f"{m['lam_bar']:.3f}"
    print(f"{100 * m['appetite']:5.1f}% {m['line']:5.0f} {m['phi']:6.3f} {m['cost']:7.2f} {m['value']:9.2f} {m['extra']:11.2f} "
          f"{100 * m['chance']:6.2f}% {100 * m['fresh']:5.2f}% {rate:>8s} {m['sigma_bar']:9.1f} {100 * m['corner']:12.2f}%")
print(f"payouts held in years 1-2 at every design: {all(m['held'] for m in MARGINS)}; the spread is guarded worst: the "
      f"4% design keeps 5% to {MARGINS[2]['sigma_bar']:.1f} against the record's bound {SIG_U:.1f}")

CORNER = law(BOOK, LAM_U, CORNER_C)
PKG_CORNER = CORNER.cheapest(A_BEST, SALE)
V_CORNER = value(BEST_Z, *BEST_P, PKG_CORNER[0], SALE)[0]
AMBIGUITY = PKG_CORNER[0] - BEST["cost"]
print(f"\nthe records' corner (rate {LAM_U:.4f}, spread {50 * CORNER_C:.1f}): the best program breaches on "
      f"{100 * MARGINS[0]['corner']:.2f}% of paths; re-designed, line {PKG_CORNER[1]:.0f} and delay {PKG_CORNER[2]:.3f}, "
      f"levers {PKG_CORNER[0]:.2f}, breach {100 * CORNER.chance(PKG_CORNER[1], PKG_CORNER[2], A_BEST, SALE):.2f}%, value "
      f"{V_CORNER:,.2f}")
CORNER_RIVALS = []
for z, p1, p2 in PRINTED[1:4]:
    pk = CORNER.cheapest(automation_profile(z, p1, p2), SALE)
    CORNER_RIVALS.append((rounds_of(z), value(z, p1, p2, pk[0], SALE)[0]))
print("  rivals at the corner (printed payouts): " + "; ".join(f"years {r}: {v:,.2f}" for r, v in CORNER_RIVALS)
      + f" -> rounds in years 1-3 stay best: {all(v < V_CORNER for _, v in CORNER_RIVALS)}")
CH11 = law(BOOK, LAM_U, SIG_U / 50.0)                     # Chapter 11's corner for the build alone
best11 = None
for line in LINES_ALL:
    d = CH11.depth_for(line, NO_CASH, 0.0)
    if d >= 0 and (DEPTH0 - d) / DPQ <= DEPTH0 / DPQ + 1e-12:
        if best11 is None or cost(line, (DEPTH0 - d) / DPQ) < best11[2]:
            best11 = (float(line), (DEPTH0 - d) / DPQ, cost(line, (DEPTH0 - d) / DPQ))
AMBIGUITY11 = best11[2] - cost(*PKG_VOL1)
print(f"the program's price of ambiguity {PKG_CORNER[0]:.2f} - {BEST['cost']:.2f} = {AMBIGUITY:.3f}; Chapter 11's for the "
      f"build alone {best11[2]:.2f} - {cost(*PKG_VOL1):.2f} = {AMBIGUITY11:.3f}")
print(f"the triggers: re-certify when the treasury's estimate of the closure rate passes {MARGINS[2]['lam_bar']:.2f}, or the "
      f"engineering office's estimate of the spread passes {MARGINS[2]['sigma_bar']:.1f} (the 4% design), or the twin's "
      f"monitor alarms; region and margins in {time.time() - t0:.1f} s")

# the feedback core's phases (Volume I, Worked Example 14.1): engineering links e, treasury links t, the new link h
RULE_RHO = 0.5 ** 0.25                                     # the board's rule: deviations halve within a year
def register(e, t, h):
    M = np.zeros((4, 4))                                   # Industrial Systems, Digital, engineering, treasury
    for a, b, w in ((0, 2, e), (1, 2, e), (0, 3, t), (1, 3, t), (0, 1, h)):
        M[a, b] = M[b, a] = w
    return M
rho_closed = lambda e, t, h: (h + math.sqrt(h * h + 8 * (e * e + t * t))) / 2
half_life = lambda r: math.log(2) / math.log(1 / r) if r < 1 else float("inf")
TRIM = math.sqrt((RULE_RHO ** 2 - 0.40 * RULE_RHO) / 4)    # every existing link trimmed so that the radius is the rule
H_LIGHT = (RULE_RHO ** 2 - 2 * (0.40 ** 2 + 0.40 ** 2)) / RULE_RHO
CORE = [("today", 0.40, 0.40, 0.0), ("phase two: link added, not repaired", 0.40, 0.40, 0.40),
        ("phase two: link run light", 0.40, 0.40, H_LIGHT), ("phase two: every link trimmed", TRIM, TRIM, 0.40),
        ("trim first, link later: after the trim", TRIM, TRIM, 0.0)]
CORE_ROWS = []
print("\nthe feedback core (Volume I's register):")
for lab, e, t, h in CORE:
    r = float(max(abs(np.linalg.eigvals(register(e, t, h)))))
    CORE_ROWS.append(dict(label=lab, e=e, t=t, h=h, rho=r, closed=rho_closed(e, t, h), half_life=half_life(r)))
    print(f"  {lab:40s} e {e:.3f}  t {t:.3f}  h {h:.3f}  radius {r:.3f}  half-life "
          + (f"{half_life(r):.2f} quarters" if r < 1 else "none: deviations grow"))
print(f"  the closed form (h + sqrt(h^2 + 8(e^2 + t^2)))/2 agrees: {all(abs(c['rho'] - c['closed']) < 1e-12 for c in CORE_ROWS)}; "
      f"the board's rule {RULE_RHO:.4f}")
print("  Proposition 16.1(ii): trim first and the core passes through "
      f"{CORE_ROWS[4]['rho']:.3f}; add the link first and it passes through {CORE_ROWS[1]['rho']:.3f}, though today and the "
      f"end state are stable; (iii) the register is symmetric, so phases each below the rule stay stable however they alternate")

LAB = Paths.draw(SEED)                                     # new draws: the lab's own seed
LAB_BEST = LAB.chance(BEST["line"], BEST["phi"], A_BEST, SALE)
LAB_RULE = rule_check(LAB, RB)
LAB_PIECES = LAB.chance(*PKG_VOL1, A5, SALE)
se = math.sqrt(ALPHA * (1 - ALPHA) / N)
print(f"\nnew draws (seed {SEED}), 100,000 paths of the same law: the signed program breaches on {100 * LAB_BEST:.2f}%, "
      f"{(LAB_BEST - ALPHA) / se:.1f} standard errors from 5% (a standard error is {100 * se:.3f} points on 100,000 paths); "
      f"round 3 on its trigger {100 * LAB_RULE[0]:.2f}%, run in year 3 on {100 * LAB_RULE[1]:.1f}%; the pieces as "
      f"certified {100 * LAB_PIECES:.2f}%")

fig, (ax, bx) = plt.subplots(1, 2, figsize=(7.6, 3.6))
ax.plot([m["extra"] for m in MARGINS], [m["sigma_bar"] for m in MARGINS], color="#1F4E79", lw=1.6, marker="o", ms=4, zorder=5)
for m in MARGINS:
    ax.text(m["extra"] + 0.25, m["sigma_bar"] - 0.3, f"{100 * m['appetite']:.1f}%", fontsize=7.5, color="#1F4E79", va="top")
for y, lab, xx, ha in ((50.0, "declared spread", 9.6, "right"), (SIG_U2, "twice the record", 0.0, "left"),
                       (SIG_U, "the record's bound", 0.0, "left")):
    ax.axhline(y, color="#999999", lw=0.7, ls=":")
    ax.text(xx, y + 0.25, lab, fontsize=7.5, color="#616161", ha=ha, va="bottom")
ax.set(xlim=(-0.3, 9.8), ylim=(49, 63), xlabel="cost of the margin (\\$ million)", ylabel="spread certified (\\$ million)")
ax.set_title("(a) what a margin of appetite certifies", fontsize=9)
bx.plot([r["trigger"] for r in RULE], [r["value"] for r in RULE], color="#1F4E79", lw=1.6, label="round 3 on a trigger")
bx.plot([RB["trigger"]], [RB["value"]], "o", color="#1F4E79", ms=4)
bx.text(RB["trigger"], RB["value"] + 0.5, f"{RB['trigger']:.0f}", fontsize=8, color="#1F4E79", ha="center", va="bottom")
bx.axhline(COMMIT3, color="#E65100", lw=1.2, ls="--", label="committed to year 3")
bx.axhline(COMMIT4, color="#2E7D32", lw=1.2, ls=":", label="committed to year 4")
bx.set(xlim=(380, 700), ylim=(1031.5, 1045.5), xlabel="trigger: cash after year 2 (\\$ million)",
       ylabel="program value (\\$ million)")
bx.set_yticks([1032, 1035, 1038, 1041, 1044], ["1,032", "1,035", "1,038", "1,041", "1,044"])
bx.legend(frameon=False, fontsize=7.5, loc="upper right")
bx.set_title("(b) keeping round 3 open", fontsize=9)
fig.suptitle("A margin buys a region; a trigger beats both timings (seed 26216)", fontsize=10.5)
plt.tight_layout(); plt.show()

## Panel 7 — Programs on one account (Worked Example 16.1; Table 16.3)

Table 16.3's rows from the panels above. Levers: the line's fees and the delay's cost; value: payouts at 10 percent plus
the year-6 savings forever, less the levers; breach: the chance of liquidity below \$350 million in some quarter of the
build, on the book's 100,000 paths. The pieces as certified keep Chapter 5's payouts and have no program value.

In [ ]:
M4 = MARGINS[2]
T163 = [("Pieces as certified", "1-3", PKG_VOL1[0], PKG_VOL1[1], cost(*PKG_VOL1), None, SEP[SALE]["chance"]),
        ("  levers to keep them", "1-3", KEEP[1], KEEP[2], KEEP[0], None, BOOK.chance(KEEP[1], KEEP[2], A5, SALE)),
        ("Best program", "1-3", BEST["line"], BEST["phi"], BEST["cost"], BEST["value"], CH_BEST),
        ("Round 3 in year 4", "1, 2, 4", PROGRAMS[1]["line"], PROGRAMS[1]["phi"], PROGRAMS[1]["cost"], PROGRAMS[1]["value"],
         BOOK.chance(PROGRAMS[1]["line"], PROGRAMS[1]["phi"], automation_profile((1, 1, 0, 1, 0, 0), 0.0, 77.0), SALE)),
        ("Round 3 on a trigger", "1, 2, 3 or 4", RB["line"], RB["phi"], RB["cost"], RB["value"], RB["chance"]),
        ("Sale after the build", "1-3", PKG_NOSALE[1], PKG_NOSALE[2], PKG_NOSALE[0], LATE[0],
         BOOK.chance(PKG_NOSALE[1], PKG_NOSALE[2], A_BEST, 0.0)),
        ("No sale", "1-3", PKG_NOSALE[1], PKG_NOSALE[2], PKG_NOSALE[0], NOSALE[0], BOOK.chance(PKG_NOSALE[1], PKG_NOSALE[2], A_BEST, 0.0)),
        ("Designed for 4%", "1-3", M4["line"], M4["phi"], M4["cost"], M4["value"], M4["chance"]),
        ("At the records' corner", "1-3", PKG_CORNER[1], PKG_CORNER[2], PKG_CORNER[0], V_CORNER,
         CORNER.chance(PKG_CORNER[1], PKG_CORNER[2], A_BEST, SALE))]
print(f"{'Program':24s} {'Rounds (years)':>15s} {'Line':>6s} {'Delay (q)':>10s} {'Levers':>8s} {'Value':>10s} {'Breach (%)':>11s}")
for name, rounds, line, phi, lev, val, ch in T163:
    print(f"{name:24s} {rounds:>15s} {line:6.0f} {phi:10.2f} {lev:8.2f} {('--' if val is None else f'{val:,.2f}'):>10s} "
          f"{100 * ch:11.1f}")
print(f"\nthe notebook ran in {time.time() - T_START:.0f} s")

## Validation — the book's numbers, and agreement with `DCT_V2_Ch16_Lab.xlsx`

Each checkpoint is a number printed in Chapter 16 (the abstract, Sections 16.1–16.4, Worked Example 16.1, Table 16.3,
Figures 16.1–16.3, the Key Takeaways, the LOS Guidance) or in Online Appendix Remark 16.B.1, Propositions 16.1–16.3 and
Example 16.B.2, with a tolerance of half a unit in its last printed digit; a few check what a theorem or proposition
promises (1 = holds). Chances are in percent. The workbook recomputes the checkpoints marked *workbook* with
spreadsheet formulas and shows PASS against the same engine values on its `Checks` sheet.

In [ ]:
def bk(s):
    # a number as the book prints it: its value and half a unit in its last printed digit
    t = s.replace(",", "").replace("−", "-")
    return float(t), 0.5 * 10.0 ** -(len(t.split(".")[1]) if "." in t else 0) + 1e-9

def c_(name, got, printed, tol=None):
    v, half = bk(printed)
    return (name, float(got), v, half if tol is None else tol)

def flag(name, ok):
    return (name, float(bool(ok)), 1.0, 0.0)

def count(name, got, n):
    return (name, float(got), float(n), 0.0)

pc = lambda x: 100.0 * x
P = {p["rounds"]: p for p in PROGRAMS}
SP, SN = SEP[SALE], SEP[0.0]
CP, CN, CG = CLAIMS["pieces"], CLAIMS["pieces_no_sale"], CLAIMS["program"]
MG = {round(100 * m["appetite"], 1): m for m in MARGINS}
CR = {r["label"]: r for r in CORE_ROWS}
checks = [
    # Step 1: the declarations, Chapter 5's plan alone, the one account (Remark 16.B.1; Example 16.B.2)
    c_("the sale's proceeds with the engineers kept (workbook)", SALE, "92"),
    c_("a dollar a year of savings, beta^6/0.10 (workbook)", SAV_PRICE, "5.645"),
    c_("Chapter 5's plan alone: value, Chapter 5 (workbook)", V5, "1021.93"),
] + [c_(f"Chapter 5's plan alone: payout in year {j + 1} (workbook)", PAY5[j], s)
     for j, s in enumerate(("85", "0", "122.5", "122.5", "122.5", "2.5"))] + [
    c_("the draw at its deepest, $m a point (workbook)", SHAPE.max(), "6.2"),
    c_("the plan without the draw relaxes toward (workbook)", 1000 * 0.62 * 112 / 100, "694"),
    c_("Volume I's package: line", PKG_VOL1[0], "170"),
    c_("Volume I's package: delay (quarters)", PKG_VOL1[1], "0.668"),
    c_("Volume I's package: levers, Table 16.3 (workbook)", cost(*PKG_VOL1), "10.44"),
    c_("the build alone: draw at quarter 7 (workbook)", BUILD_ALONE["draw"], "121.6"),
    c_("the build alone: closure buffer (workbook)", BUILD_ALONE["buffer"], "16.9"),
    count("the build alone: its deepest quarter (workbook)", BUILD_ALONE["quarter"], 7),
    # Step 2: check jointly (Theorem 16.1(i); Figure 16.1; Table 16.3; Example 16.B.2)
    c_("pieces as certified, with the sale: breach, %", pc(SP["chance"]), "74.33"),
    c_("pieces as certified, with the sale: breach, %, Step 2 and Table 16.3", pc(SP["chance"]), "74.3"),
    c_("pieces as certified: breach, %, the abstract and Key Takeaways", pc(SP["chance"]), "74"),
    c_("pieces as certified, with the sale: fresh paths, %", pc(SP["fresh"]), "74.20"),
    c_("pieces as certified, with the sale: fresh paths, %, Step 2", pc(SP["fresh"]), "74.2"),
    c_("pieces as certified, without the sale: breach, %", pc(SN["chance"]), "99.58"),
    c_("pieces as certified, without the sale: breach, %, Step 2", pc(SN["chance"]), "99.6"),
    c_("pieces as certified, without the sale: fresh paths, %", pc(SN["fresh"]), "99.57"),
    c_("pieces as certified, no levers, without the sale, %", pc(SN["no_levers"]), "99.90"),
    c_("pieces as certified, no levers, with the sale, %", pc(SP["no_levers"]), "87.29"),
    c_("planned margin at quarter 9, without the sale (workbook)", SN["margins"][8], "-84.7"),
    c_("planned margin at quarter 9, with the sale (workbook)", SP["margins"][8], "7.3"),
    c_("planned margin at quarter 9, with the sale, Step 2 (workbook)", SP["margins"][8], "7"),
    count("the binding quarter of the joint check (workbook)", CP["quarter"], 9),
    c_("claims at quarter 9, pieces: the build's draw (workbook)", CP["build"], "115.4"),
    c_("claims at quarter 9, pieces: the rounds less their savings (workbook)", CP["rounds"], "154.4"),
    c_("claims at quarter 9, pieces: the payouts made (workbook)", CP["payouts"], "85.0"),
    c_("claims at quarter 9, pieces: the closure buffer (workbook)", CP["buffer"], "16.9"),
    c_("claims at quarter 9, pieces with the sale: left of 362 (workbook)", CP["left"], "-9.6"),
    c_("claims at quarter 9, pieces without the sale: left of 270 (workbook)", CN["left"], "-101.6"),
    c_("claims at quarter 9, program: the build's draw (workbook)", CG["build"], "80.8"),
    c_("claims at quarter 9, program: the rounds less their savings (workbook)", CG["rounds"], "154.4"),
    c_("claims at quarter 9, program: the payouts made (workbook)", CG["payouts"], "0"),
    c_("claims at quarter 9, program: the closure buffer (workbook)", CG["buffer"], "26.9"),
    c_("claims at quarter 9, program: left of 362 (workbook)", CG["left"], "100.0"),
    c_("the room with the sale (workbook)", CP["room"], "362"),
    c_("levers to keep the pieces: line", KEEP[1], "165"),
    c_("levers to keep the pieces: delay", KEEP[2], "5.634"),
    c_("levers to keep the pieces: delay, Table 16.3", KEEP[2], "5.63"),
    c_("levers to keep the pieces: delay, Step 2", KEEP[2], "5.6"),
    c_("levers to keep the pieces: cost (workbook)", KEEP[0], "50.02"),
    c_("levers to keep the pieces: cost, Step 2 (workbook)", KEEP[0], "50.0"),
    c_("levers to keep the pieces: breach, %, Table 16.3", pc(BOOK.chance(KEEP[1], KEEP[2], A5, SALE)), "5.0"),
    flag("no package keeps the pieces without the sale", SEP[0.0]["cheapest"] is None and best_depth < 0),
    # Step 3: order (Theorem 16.2; Figure 16.2(a); Table 16.3; Example 16.B.2)
] + [c_(f"rounds in years {r}: value (workbook)", P[r]["value"], s) for r, s in
     (("1, 2, 3", "1039.41"), ("1, 2, 4", "1036.80"), ("1, 3, 4", "1022.57"), ("1, 4, 5", "1001.59"), ("2, 3, 4", "953.58"),
      ("2, 4, 5", "937.60"), ("3, 4, 5", "883.16"), ("4, 5, 6", "821.11"))] + [
    c_("rounds in years 1, 2, 4: payout in year 2", P["1, 2, 4"]["p2"], "77"),
    c_("rounds in years 1, 2, 4: line", P["1, 2, 4"]["line"], "160"),
    c_("rounds in years 1, 2, 4: delay", P["1, 2, 4"]["phi"], "1.477"),
    c_("rounds in years 1, 2, 4: delay, Table 16.3", P["1, 2, 4"]["phi"], "1.48"),
    c_("rounds in years 1, 2, 4: levers (workbook)", P["1, 2, 4"]["cost"], "16.62"),
    c_("rounds in years 1, 3, 4: payout in year 1", P["1, 3, 4"]["p1"], "35"),
    c_("rounds in years 1, 3, 4: line", P["1, 3, 4"]["line"], "140"),
    c_("rounds in years 1, 3, 4: delay", P["1, 3, 4"]["phi"], "0.012"),
    c_("rounds in years 1, 3, 4: levers (workbook)", P["1, 3, 4"]["cost"], "4.29"),
    flag("every printed timing is the best continuation of its first three years", all(p["best_tail"] for p in PROGRAMS)),
    c_("round 3 moved to year 4 costs, Step 3 (workbook)", BEST["value"] - P["1, 2, 4"]["value"], "2.6"),
    c_("rounds started in year 2 cost, Step 3 (workbook)", BEST["value"] - P["2, 3, 4"]["value"], "85.8"),
    c_("no sale: line", PKG_NOSALE[1], "165"),
    c_("no sale: delay", PKG_NOSALE[2], "5.97"),
    c_("no sale: levers (workbook)", PKG_NOSALE[0], "52.71"),
    c_("no sale: value (workbook)", NOSALE[0], "938.95"),
    c_("sale after the build: value (workbook)", LATE[0], "1001.79"),
    c_("the sale first is worth, Step 3 (workbook)", SALE_WORTH, "100.5"),
    c_("the sale's proceeds paid out at the end of year 3 (workbook)", SALE * BY ** 3, "69.1"),
    c_("the levers the sale makes unnecessary (workbook)", PKG_NOSALE[0] - BEST["cost"], "31.3"),
    c_("the sale first beats the sale after the build by (workbook)", BEST["value"] - LATE[0], "37.6"),
    # Step 4: price and allocate (Theorem 16.1(iii); Proposition 16.3; Figure 16.2(b); Example 16.B.2)
    c_("best program: line", BEST["line"], "160"),
    c_("best program: delay", BEST["phi"], "2.071"),
    c_("best program: delay, Table 16.3", BEST["phi"], "2.07"),
    c_("best program: levers (workbook)", BEST["cost"], "21.37"),
    c_("best program: levers, the abstract (workbook)", BEST["cost"], "21.4"),
    c_("best program: value, the abstract", BEST["value"], "1039.4"),
    c_("best program: breach, %", pc(CH_BEST), "5.00"),
    c_("best program: fresh paths, %", pc(CH_BEST_F), "4.99"),
    c_("best program: payout in year 1", BEST["payouts"][0], "0"),
    c_("best program: payout in year 2", BEST["payouts"][1], "0"),
] + [c_(f"best program: payout in year {j + 3} (workbook)", BEST["payouts"][j + 2], s)
     for j, s in enumerate(("189.9", "160.4", "158.8", "32.7"))] + [
    flag("the best program is best nearby: payouts and lines", LOCAL_BEST),
    c_("the appetite at 4.5%: value (workbook)", APP_LO["value"], "1038.43"),
    c_("the appetite at 5.5%: value (workbook)", APP_HI["value"], "1040.25"),
    c_("the appetite's price, $m a point (workbook)", MU, "1.83"),
    c_("the appetite's price, the board takeaway (workbook)", MU, "1.8"),
    c_("Volume I's price for the build alone (workbook)", MU_VOL1, "1.81"),
    flag("payouts held at 4.5% and 5.5%", APP_LO["held"] and APP_HI["held"]),
    c_("the line buys a point of chance for (workbook)", LEVER["line"], "1.76"),
    c_("the delay buys a point of chance for (workbook)", LEVER["delay"], "2.29"),
    c_("paying in year 1 earns a point (workbook)", LEVER["pay1"], "0.77"),
    c_("paying in year 2 earns a point (workbook)", LEVER["pay2"], "0.67"),
    c_("beta - beta^3 (workbook)", BY - BY ** 3, "0.158"),
    c_("points of chance a dollar paid in year 1 spends (workbook)", 100 * (CHANCES["pay1"] - c0) / dp, "0.204"),
    c_("a dollar of room at the start (workbook)", ROOM_DOLLAR, "1.088"),
    c_("a dollar of room, Section 16.3 (workbook)", ROOM_DOLLAR, "1.09"),
    c_("a dollar of room: paid out at the end of year 3 (workbook)", BY ** 3, "0.75"),
    c_("a dollar of room: levers made unnecessary (workbook)", BEST["cost"] - ROOM1["cost"], "0.34"),
    c_("a point of capability at Chapter 7's price (workbook)", CAP_PRICE, "3.90"),
    c_("capability of rounds in years 1-3 (workbook)", CAP_CREDIT["1, 2, 3"], "10.67"),
    c_("capability of rounds in years 1-3, Section 16.3 (workbook)", CAP_CREDIT["1, 2, 3"], "10.7"),
    c_("capability of rounds in years 1, 2, 4 (workbook)", CAP_CREDIT["1, 2, 4"], "10.38"),
    flag("levers bought near the appetite's price, payouts below it", LEVER["pay1"] < MU < LEVER["delay"] and LEVER["pay2"] < MU),
    # Step 5: keep a choice open (Theorem 16.2(iii); Proposition 16.2; Figure 16.3(b); Example 16.B.2)
    c_("round 3 on a trigger: trigger (workbook)", RB["trigger"], "490"),
    c_("round 3 on a trigger: line", RB["line"], "155"),
    c_("round 3 on a trigger: delay", RB["phi"], "1.642"),
    c_("round 3 on a trigger: delay, Step 5 and Table 16.3", RB["phi"], "1.64"),
    c_("round 3 on a trigger: levers (workbook)", RB["cost"], "17.79"),
    c_("round 3 on a trigger: year 3 on, % of paths", pc(RB["share_run"]), "83.0"),
    c_("round 3 on a trigger: year 3 on, %, Step 5", pc(RB["share_run"]), "83"),
    c_("round 3 on a trigger: year 3 on, fresh paths, %", pc(RB_FRESH[1]), "83.0"),
    c_("round 3 on a trigger: value (workbook)", RB["value"], "1040.76"),
    c_("round 3 on a trigger: more than committing it to year 3 (workbook)", RB["value"] - COMMIT3, "1.35"),
    c_("round 3 on a trigger: breach, %", pc(RB["chance"]), "5.00"),
    c_("round 3 on a trigger: fresh paths, %", pc(RB_FRESH[0]), "5.00"),
    flag("Theorem 16.2(iii): the rule beats both committed timings (workbook)", RB["value"] > max(COMMIT3, COMMIT4)),
    c_("Proposition 16.2: the project started now (workbook)", N0, "5.00"),
    c_("Proposition 16.2: the project if it waits (workbook)", N1, "22.73"),
    c_("Proposition 16.2: the expected loss waiting avoids (workbook)", loss_avoided, "20"),
    c_("Proposition 16.2: what waiting costs (workbook)", wait_cost, "2.27"),
    # Step 6: certify (Theorem 16.3; Proposition 16.1; Figure 16.3(a); Example 16.B.2)
    c_("design for 5%: closure rate certified", MG[5.0]["lam_bar"], "0.100"),
    c_("design for 5%: spread certified", MG[5.0]["sigma_bar"], "50.0"),
    c_("design for 5%: breach at the records' corner, %", pc(MG[5.0]["corner"]), "13.88"),
    c_("design for 5%: breach at the corner, %, Step 6", pc(MG[5.0]["corner"]), "13.9"),
    flag("Theorem 16.3(i): no slack: a higher rate or spread breaks the appetite",
         law(BOOK, 0.105, 1.0).chance(BEST["line"], BEST["phi"], A_BEST, SALE) > ALPHA
         and law(BOOK, LAM, 1.01).chance(BEST["line"], BEST["phi"], A_BEST, SALE) > ALPHA),
] + [c_(f"design for {k:g}%: {lab}{' (workbook)' if lab in ('levers', 'value') else ''}",
        pc(MG[k][key]) if key == "corner" else MG[k][key], s) for k, vals in
     ((4.5, ("22.35", "1038.43", "51.0", "12.82")), (4.0, ("23.44", "1037.34", "52.2", "11.85")),
      (3.5, ("24.65", "1036.13", "53.5", "10.84")), (3.0, ("26.02", "1034.76", "55.0", "9.69")),
      (2.5, ("27.50", "1033.28", "56.5", "8.65")), (2.0, ("29.30", "1031.48", "58.4", "7.58")))
     for lab, key, s in zip(("levers", "value", "spread certified", "breach at the corner, %"),
                            ("cost", "value", "sigma_bar", "corner"), vals)] + [
    c_("design for 4.5%: closure rate certified", MG[4.5]["lam_bar"], "0.192"),
    c_("design for 4%: closure rate certified", MG[4.0]["lam_bar"], "0.298"),
    c_("design for 4%: closure rate, Step 6 and Section 16.4", MG[4.0]["lam_bar"], "0.30"),
    flag("designs for 3.5% and below certify a closure rate of 0.40 or more", all(MG[k]["capped"] for k in (3.5, 3.0, 2.5, 2.0))),
    c_("design for 4%: the margin's cost (workbook)", MG[4.0]["extra"], "2.07"),
    c_("design for 4%: a point of margin, the board takeaway (workbook)", MG[4.0]["extra"], "2.1"),
    c_("design for 3%: the margin's cost (workbook)", MG[3.0]["extra"], "4.65"),
    c_("design for 4%: line, Table 16.3", MG[4.0]["line"], "165"),
    c_("design for 4%: delay, Table 16.3", MG[4.0]["phi"], "2.31"),
    c_("design for 4%: breach, %, Table 16.3", pc(MG[4.0]["chance"]), "4.0"),
    c_("the records' corner: closure rate (workbook)", LAM_U, "0.315"),
    c_("the records' corner: spread", 50 * CORNER_C, "61.6"),
    c_("the record twice as long: spread bound, Figure 16.3(a) (workbook)", SIG_U2, "57.6"),
    c_("at the corner: line", PKG_CORNER[1], "200"),
    c_("at the corner: delay", PKG_CORNER[2], "3.440"),
    c_("at the corner: delay, Step 6", PKG_CORNER[2], "3.44"),
    c_("at the corner: levers (workbook)", PKG_CORNER[0], "33.52"),
    c_("at the corner: value (workbook)", V_CORNER, "1027.26"),
    c_("at the corner: breach, %, Table 16.3", pc(CORNER.chance(PKG_CORNER[1], PKG_CORNER[2], A_BEST, SALE)), "5.0"),
    flag("at the corner, rounds in years 1-3 stay best", all(v < V_CORNER for _, v in CORNER_RIVALS)),
    c_("the program's price of ambiguity, 33.52 less 21.37 (workbook)", AMBIGUITY, "12.14"),
    c_("Chapter 11's price of ambiguity for the build alone (workbook)", AMBIGUITY11, "13.56"),
    c_("the core today (workbook)", CR["today"]["rho"], "0.800"),
    c_("the core with the new link at 0.40 (workbook)", CR["phase two: link added, not repaired"]["rho"], "1.025"),
    c_("the core with every link trimmed (workbook)", CR["phase two: every link trimmed"]["rho"], "0.841"),
    c_("every existing link trimmed to (workbook)", TRIM, "0.304"),
    c_("trim first: the core passes through (workbook)", CR["trim first, link later: after the trim"]["rho"], "0.609"),
    c_("trim first, Section 16.2 (workbook)", CR["trim first, link later: after the trim"]["rho"], "0.61"),
    c_("trim first: half-life, quarters (workbook)", CR["trim first, link later: after the trim"]["half_life"], "1.40"),
    c_("the link with the trim: half-life, quarters (workbook)", CR["phase two: every link trimmed"]["half_life"], "4.00"),
    c_("the link run light (workbook)", H_LIGHT, "0.080"),
    c_("the link run light keeps the radius at (workbook)", CR["phase two: link run light"]["rho"], "0.841"),
    flag("Proposition 16.1(ii): stable start and end, an unstable phase between",
         CR["today"]["rho"] < 1 < CR["phase two: link added, not repaired"]["rho"]
         and CR["phase two: every link trimmed"]["rho"] < 1),
    # Table 16.3 and the Key Takeaways
    c_("Table 16.3: pieces as certified, delay", PKG_VOL1[1], "0.67"),
    c_("Table 16.3: the best program's value, Key Takeaways", BEST["value"], "1039.41"),
    c_("Table 16.3: the best program's levers, Key Takeaways", BEST["cost"], "21.37"),
    c_("Table 16.3: round 3 on a trigger, breach, %", pc(RB["chance"]), "5.0"),
    c_("Table 16.3: sale after the build, breach, %", pc(T163[5][6]), "5.0"),
    c_("Table 16.3: no sale, breach, %", pc(T163[6][6]), "5.0"),
    c_("Table 16.3: round 3 in year 4, breach, %", pc(T163[3][6]), "5.0"),
    c_("Table 16.3: designed for 4%, value", M4["value"], "1037.34"),
]
ok = 0
for name, got, want, tol in checks:
    status = "PASS" if abs(got - want) <= tol else "FAIL"
    ok += status == "PASS"
    print(f"{name:78s} {got:12.4f}   book {want:10.4f}   {status}")
print(f"\n{ok} of {len(checks)} checkpoints PASS")

**Next.** This is the last chapter of Volume II, and of the series. Volume I drew the map: the seven moves of DCT,
from Represent to Adapt, run as a loop, and the gates that tell a board when it is ready to optimize. Volume II supplied
the methods, each piece of Meridian's transformation optimized and certified on its own terms, and this chapter signed
them as one program on one account, with its roadmap, its prices, its certificate chain, its region and its triggers.
The program is not signed once: when an estimate leaves the region, a monitor alarms, or the board changes a
declaration, the loop runs again, and the program is re-certified for the next cycle. AXIOM-16 runs the same program
interactively (Lab 2.16); this notebook and the workbook are its reproducible record.